# 06 · Session layer: brute force, ID enumeration and scanning

Some attacks look normal one request at a time: one failed login, one product ID, one missing page. They only show up when we look at what one client (IP) does over time. `ml/sessions.py` groups requests per IP into sessions (a new session after 30 minutes of silence) and applies simple rules:

| behaviour | rule |
|---|---|
| brute force | at least 3 failed logins (401/403 on a login path), or at least 6 login attempts |
| ID enumeration | at least 15 different numeric IDs of the same resource, e.g. `/api/BasketItems/{id}` |
| scan | at least 3 probes for hidden files (`/.env`, `/.git`, `*.bak` ...) or at least 10 not-found answers |

Each session also gets a **risk score** (0 to 100) with the reasons behind it.

**Why rules and not a trained model:** we have no public dataset with labelled sessions, and training on our generator would teach the model the generator itself (see notebook 05). Rules with clear thresholds are also easy to explain in an investigation.

**Data:** run3 is development data: the thresholds were chosen on it, so these numbers are optimistic. The honest test is run4 and a third site, later. Normal users in run3 include people who mistype their password once or twice, open a few product IDs and hit a missing page.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ml.detector import build_pipeline, has_payload, to_frame
from ml.sessions import analyze_sessions
from eval.match_labels import match

csic = pd.read_csv(ROOT / 'data' / 'processed' / 'csic_clean.csv', keep_default_na=False)
y = csic['label'].values
X = to_frame(csic['Method'], csic['path'], csic['body'])
keep = (y == 0) | has_payload(X).values
HAND = [c for c in X.columns if c != 'text']
model = build_pipeline(HAND, pos_weight=(y[keep] == 0).sum() / (y[keep] == 1).sum()).fit(X[keep], y[keep])

run3, report = match(ROOT / 'data' / 'raw' / 'juiceshop_generated_run3.log',
                     ROOT / 'data' / 'labels' / 'requests_run3.csv')
print('matching:', report)
run3['pred'] = model.predict(to_frame(run3['method'], run3['path'], run3['body']))
sessions, requests = analyze_sessions(run3)
print(len(sessions), 'sessions')

matching: {'log_lines': 4397, 'labels': 4397, 'matched': 4397, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


300 sessions


## 1. Are the attackers found? (one row per session)

In [2]:
truth = requests.groupby('session')['attack_type'].agg(lambda s: ','.join(sorted(set(s) - {''})) or 'normal')
sessions['truth'] = sessions['session'].map(truth)
pd.crosstab(sessions['truth'], sessions['risk_level'])

risk_level,high,medium,none
truth,,,
bruteforce,0,10,0
cmdi,10,6,0
enum,5,14,0
normal,0,0,210
scan,12,0,0
sqli,6,6,0
traversal,11,0,0
xss,1,9,0


Every attacker session gets a risk level, and no normal session does. The behaviour the rules name:

In [3]:
pd.crosstab(sessions['truth'], sessions['behaviours'].map(lambda b: ', '.join(b) or '(payload only)'))

behaviours,(payload only),bruteforce,enumeration,"enumeration, scan",scan
truth,,,,,
bruteforce,0,10,0,0,0
cmdi,16,0,0,0,0
enum,0,0,14,5,0
normal,210,0,0,0,0
scan,0,0,0,0,12
sqli,12,0,0,0,0
traversal,0,0,0,0,11
xss,9,1,0,0,0


- Payload attacks (SQLi, XSS, traversal, command injection) are found by the per-request model; the session adds their risk.
- Traversal sessions are also named *scan*, because they ask for files like `/etc/passwd`.
- Some enumeration sessions are also named *scan*, because most of the IDs they try do not exist (404).

## 2. Request level: per-request model alone vs. model + session layer

In [4]:
requests['final'] = requests['pred'] | requests['session_attack']
is_atk = requests['label'] != 'normal'
by_type = requests[is_atk].groupby('attack_type')[['pred', 'final']].mean().round(3)
by_type.columns = ['per-request model', 'model + session layer']
by_type

,per-request model,model + session layer
attack_type,,
bruteforce,0.000,1.000
cmdi,0.907,0.907
enum,0.000,1.000
scan,0.413,1.000
sqli,0.974,0.974
traversal,1.000,1.000
xss,1.000,1.000


In [5]:
print(f"attacks caught: {requests.loc[is_atk, 'final'].mean():.1%}")
print(f"false alarms:   {int(requests.loc[~is_atk, 'final'].sum())} of {int((~is_atk).sum())} normal requests")

attacks caught: 99.0%
false alarms:   0 of 3328 normal requests


## 3. What an investigation looks like

The riskiest sessions, with the reasons the rules give.

In [6]:
cols = ['ip', 'start', 'requests', 'attack_requests', 'behaviours', 'risk_score', 'risk_level', 'reasons']
sessions[cols].head(8)

,ip,start,requests,attack_requests,behaviours,risk_score,risk_level,reasons
0,10.1.0.54,2026-10-19 23:51:23+00:00,9,8,[bruteforce],100,high,"[8 requests carry an attack payload, 3 failed ..."
1,10.1.0.103,2026-10-19 23:51:55+00:00,5,4,[scan],90,high,"[4 requests carry an attack payload, 4 probes ..."
2,10.1.0.104,2026-10-19 23:51:55+00:00,7,6,[scan],90,high,"[6 requests carry an attack payload, 6 probes ..."
3,10.1.0.107,2026-10-19 23:51:56+00:00,18,8,[scan],90,high,"[8 requests carry an attack payload, 8 probes ..."
4,10.1.0.125,2026-10-19 23:52:06+00:00,19,8,[scan],90,high,"[8 requests carry an attack payload, 8 probes ..."
5,10.1.0.138,2026-10-19 23:52:14+00:00,5,4,[scan],90,high,"[4 requests carry an attack payload, 4 probes ..."
6,10.1.0.152,2026-10-19 23:52:23+00:00,21,8,[scan],90,high,"[8 requests carry an attack payload, 8 probes ..."
7,10.1.0.16,2026-10-19 23:50:58+00:00,8,7,[scan],90,high,"[7 requests carry an attack payload, 7 probes ..."


## 4. Limitations

- **Brute force below 3 attempts** looks like a user who mistyped the password, so it is not flagged.
- **Sites that answer a failed login with 200 or a redirect:** only the *6 login attempts* rule applies.
- **Enumeration with fewer than 15 IDs**, or with IDs that are not numbers (UUIDs, names), is not flagged. A real shopper who opens 15+ products in one visit would be flagged as enumeration (medium risk).
- **Scans** on sites that answer every unknown page with 200 (like Juice Shop) are only found through the hidden-file probes.
- **One IP shared by many people** (an office or a mobile network) adds their requests together.
- Thresholds were set on run3, so these numbers will be re-checked on run4 and a third site.